# Tailoring Benchmark — driver & results explorer

Drives the **tailoring efficacy benchmark** (issue #51) and visualizes its results.

The benchmark replays the exact user flow through the web API — register → upload
resume → create job → paste JD → **Analyze** → **Tailor** → **Export** — for every
job description in `eval/jd_dataset/`, on an isolated temp database. Per task it
measures the three observed failure modes plus ATS efficacy:

| Family | Question it answers |
|---|---|
| `ats` | Does tailoring raise the composite ATS score, and which component moves? |
| `experience_allocation` | Does the amount of text an experience gets track its JD relevance? |
| `skills` | Is the skills section selective, organized, and does it keep the matched skills? |
| `redundancy` | Are the same skill terms over-repeated across the resume? |

**Modes** — `STUB = True` runs fully offline behind a deterministic fake LLM
(pipeline plumbing + deterministic guarantees only). `STUB = False` uses the real
LLMs (needs `ANTHROPIC_API_KEY`/`OPENAI_API_KEY`) and is the number that matters.

Requirements: `pip install notebook` if Jupyter is missing; matplotlib + pandas ship
with the repo venv.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "eval":          # allow running from eval/ or repo root
    ROOT = ROOT.parent
RESULTS_DIR = ROOT / "eval" / "results"

RUN_BENCHMARK = False   # True → run a fresh benchmark below (else: load latest results)
STUB = False            # True → offline deterministic stub LLM
LIMIT = 0               # 0 = all tasks

def latest_results_path():
    files = sorted(RESULTS_DIR.glob("tailoring_benchmark_*.json"))
    if not files:
        raise FileNotFoundError(
            "No results yet - set RUN_BENCHMARK = True and re-run this notebook, "
            "or run: python eval/tailoring_benchmark.py --stub"
        )
    return files[-1]

In [ ]:
# Run the benchmark (subprocess: it isolates its own env/DB before importing the app)
if RUN_BENCHMARK:
    cmd = [sys.executable, str(ROOT / "eval" / "tailoring_benchmark.py")]
    if STUB:
        cmd.append("--stub")
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])

In [ ]:
results_path = latest_results_path()
results = json.loads(results_path.read_text(encoding="utf-8"))
print(f"run: {results['timestamp']}  mode: {results['mode']}  "
      f"tasks: {results['dataset_size']}  failed: {results['failed'] or 'none'}")

ok = [t for t in results["task_results"] if "error" not in t]

def dig(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

df = pd.DataFrame([{
    "task": t["task_id"],
    "company": t["company"],
    "baseline": dig(t, "metrics", "ats", "baseline_composite"),
    "tailored": dig(t, "metrics", "ats", "tailored_composite"),
    "delta": dig(t, "metrics", "ats", "delta"),
    "alloc_corr": dig(t, "metrics", "experience_allocation", "allocation_correlation"),
    "skills_rendered": dig(t, "metrics", "skills", "rendered_count"),
    "matched_recall": dig(t, "metrics", "skills", "matched_recall"),
    "selection_ratio": dig(t, "metrics", "skills", "selection_ratio"),
    "max_repeat": dig(t, "metrics", "redundancy", "max_term_repetition"),
    "over_repeated": dig(t, "metrics", "redundancy", "over_repeated_count"),
    "bullet_ttr": dig(t, "metrics", "redundancy", "bullet_type_token_ratio"),
} for t in ok]).set_index("task")

df  # full per-task table (also the accessible fallback for every chart below)

## Aggregate view

Every chart reads left→right as "how did this run do"; the table above is the
exact-number fallback. Colors: blue = tailored/primary measure, violet = baseline
context, red = a negative value that needs attention.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Reference palette (validated): see the dataviz palette notes in the repo history.
SURFACE   = "#fcfcfb"
INK       = "#0b0b0b"
INK_2     = "#52514e"
GRID      = "#e5e4e0"
BLUE      = "#2a78d6"   # primary / tailored / positive
VIOLET    = "#4a3aa7"   # baseline context
RED       = "#e34948"   # negative pole (diverging)

def style_ax(ax, xgrid=False, ygrid=True):
    ax.set_facecolor(SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_2, labelsize=9)
    ax.xaxis.grid(xgrid, color=GRID, linewidth=0.8)
    ax.yaxis.grid(ygrid, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

def short(task_id, n=26):
    return task_id if len(task_id) <= n else task_id[: n - 1] + "…"

labels = [short(t) for t in df.index]

In [ ]:
# ATS composite: baseline vs tailored, per task
import numpy as np

fig, ax = plt.subplots(figsize=(9, 0.55 * len(df) + 1.2), facecolor=SURFACE)
y = np.arange(len(df))
h = 0.34
ax.barh(y - h / 2 - 0.02, df["baseline"], height=h, color=VIOLET, label="baseline")
ax.barh(y + h / 2 + 0.02, df["tailored"], height=h, color=BLUE, label="tailored")
for yi, (b, t) in enumerate(zip(df["baseline"], df["tailored"])):
    if pd.notna(t):
        ax.text(t + 1, yi + h / 2 + 0.02, f"{t:.0f}", va="center", fontsize=8, color=INK_2)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlim(0, 100)
style_ax(ax, xgrid=True, ygrid=False)
ax.set_title("ATS composite - baseline vs tailored", color=INK, fontsize=12, loc="left")
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
plt.tight_layout()
plt.show()

agg = results["aggregate"]
print("mean delta:", dig(agg, "ats_delta", "mean"),
      " median:", dig(agg, "ats_delta", "median"))

In [ ]:
# The three quality families, one small multiple each (independent axes by design)
fig, axes = plt.subplots(1, 3, figsize=(13, 0.5 * len(df) + 1.6), facecolor=SURFACE)

# 1) allocation correlation — diverging around 0 (positive = text follows relevance)
vals = df["alloc_corr"].fillna(0)
axes[0].barh(range(len(df)), vals, height=0.5,
             color=[BLUE if v >= 0 else RED for v in vals])
axes[0].axvline(0, color=INK_2, linewidth=1)
axes[0].set_xlim(-1.05, 1.05)
axes[0].set_title("Experience allocation vs relevance\n(Spearman, 1.0 = ideal)",
                  color=INK, fontsize=10, loc="left")

# 2) skills rendered, with the scorer's cap bounds shaded
from agents.skill_scorer import MAX_SKILLS, MIN_SKILLS
axes[1].axvspan(MIN_SKILLS, MAX_SKILLS, color=GRID, alpha=0.5, zorder=0)
axes[1].barh(range(len(df)), df["skills_rendered"], height=0.5, color=BLUE)
axes[1].set_title(f"Skills rendered\n(band = cap bounds {MIN_SKILLS}-{MAX_SKILLS})",
                  color=INK, fontsize=10, loc="left")
axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))

# 3) redundancy — count of terms mentioned more than the stuffing threshold
from eval.metrics import OVER_REPEAT_THRESHOLD
axes[2].barh(range(len(df)), df["over_repeated"], height=0.5, color=BLUE)
axes[2].set_title(f"Over-repeated terms\n(> {OVER_REPEAT_THRESHOLD} mentions; 0 = clean)",
                  color=INK, fontsize=10, loc="left")
axes[2].set_xlim(0, max(df["over_repeated"].max(), 3))
axes[2].xaxis.set_major_locator(MaxNLocator(integer=True))

for i, ax in enumerate(axes):
    ax.set_yticks(range(len(df)), labels if i == 0 else [""] * len(df))
    ax.invert_yaxis()
    style_ax(ax, xgrid=True, ygrid=False)
plt.tight_layout()
plt.show()

## Per-task drill-down: where did the text go?

Pick a task and see each experience's JD relevance next to the share of bullet
text it received. A well-balanced tailoring gives more words to more relevant
experiences (bars descending together).

In [ ]:
TASK = df.index[0]   # <- change to any task id from the table above

task = next(t for t in ok if t["task_id"] == TASK)
rows = task["metrics"]["experience_allocation"]["experiences"]
alloc = pd.DataFrame(rows)
display(alloc)

fig, ax = plt.subplots(figsize=(8, 0.6 * len(alloc) + 1), facecolor=SURFACE)
y = np.arange(len(alloc))
h = 0.34
rel = alloc["relevance"] / max(alloc["relevance"].max(), 1e-9)
ax.barh(y - h / 2 - 0.02, rel, height=h, color=VIOLET, label="relevance (normalized)")
ax.barh(y + h / 2 + 0.02, alloc["word_share"], height=h, color=BLUE, label="share of bullet words")
ax.set_yticks(y, [f"{r.title} - {r.company}" for r in alloc.itertuples()])
ax.invert_yaxis()
style_ax(ax, xgrid=True, ygrid=False)
ax.set_title(f"Text allocation vs relevance - {TASK}", color=INK, fontsize=11, loc="left")
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
plt.tight_layout()
plt.show()

corr = task["metrics"]["experience_allocation"]["allocation_correlation"]
over = task["metrics"]["redundancy"]["over_repeated"]
print("allocation correlation:", corr)
print("over-repeated terms:", over or "none")

## Resume viewer

Renders the tailored output of any task as readable Markdown, straight from the
run's persisted artifacts (`eval/results/renders/<ts>/<task>.json`). The exported
LaTeX source sits next to it as `<task>.tex` — compile it with tectonic for the
pixel-final PDF.

In [ ]:
from IPython.display import Markdown, display

RENDERS = RESULTS_DIR / "renders" / results["timestamp"]

def show_resume(task_id):
    content = json.loads((RENDERS / f"{task_id}.json").read_text(encoding="utf-8"))
    lines = [f"### Tailored resume - `{task_id}`", ""]
    order = content.get("_section_order") or ["experience", "projects", "skills"]
    for section in order:
        if section == "experience" and content.get("experiences"):
            lines.append("#### Experience")
            for e in content["experiences"]:
                lines.append(f"**{e.get('title','')}** - {e.get('company','')} "
                             f"({e.get('start_date','')} to {e.get('end_date','')})")
                lines += [f"- {b}" for b in e.get("bullets") or []]
                lines.append("")
        elif section == "projects" and content.get("projects"):
            lines.append("#### Projects")
            for p in content["projects"]:
                lines.append(f"**{p.get('name','')}**")
                lines += [f"- {b}" for b in p.get("bullets") or []]
                lines.append("")
        elif section == "skills":
            ranked = content.get("skills_ranked") or []
            if ranked:
                lines.append("#### Technical Skills (JD-ranked)")
                by_cat = {}
                for s in ranked:
                    by_cat.setdefault(s.get("category") or "Other", []).append(s["name"])
                for cat, names in by_cat.items():
                    lines.append(f"- **{cat}:** {', '.join(names)}")
                lines.append("")
    display(Markdown("\n".join(lines)))
    print("LaTeX source:", RENDERS / f"{task_id}.tex")

show_resume(TASK)

## Trend across runs

Every benchmark run persists an artifact, so pipeline changes show up as steps in
these lines. Run the benchmark before and after a change to see its effect.

In [ ]:
runs = []
for path in sorted(RESULTS_DIR.glob("tailoring_benchmark_*.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    runs.append({
        "run": r["timestamp"],
        "mode": r["mode"],
        "mean_delta": dig(r["aggregate"], "ats_delta", "mean"),
        "mean_alloc_corr": dig(r["aggregate"], "allocation_correlation", "mean"),
        "mean_over_repeated": dig(r["aggregate"], "over_repeated_count", "mean"),
    })
trend = pd.DataFrame(runs)
display(trend)

measures = [("mean_delta", "Mean ATS delta"),
            ("mean_alloc_corr", "Mean allocation correlation"),
            ("mean_over_repeated", "Mean over-repeated terms")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3), facecolor=SURFACE)
x = range(len(trend))
for ax, (colname, title) in zip(axes, measures):
    ax.plot(x, trend[colname], color=BLUE, linewidth=2, marker="o", markersize=6)
    ax.set_xticks(list(x), trend["run"], rotation=45, ha="right", fontsize=7)
    ax.set_title(title, color=INK, fontsize=10, loc="left")
    style_ax(ax)
plt.tight_layout()
plt.show()